<a href="https://colab.research.google.com/github/Neoseb92/Modulo-1-AI/blob/main/Notebook_Nox_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook 2 — Matriz de correlación, VIF, matriz H, leverage y Cook (SIMAT 2026)

Datos horarios: https://aire.cdmx.gob.mx/descargas/Opendata/anuales_horarios/contaminantes_2026.csv

## 0. Librerías

In [4]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor

## 1. Cargar los datos

In [5]:
ruta = "contaminantes_2026(3).csv"
df = pd.read_csv(ruta, skiprows=9, encoding="utf-8")
df["date"] = pd.to_datetime(df["date"])
df.head()

,date,id_station,id_parameter,valor,unit
0,2026-01-01,ACO,CO,NaN,15
1,2026-01-01,ACO,NO,NaN,1
2,2026-01-01,ACO,NO2,NaN,1
3,2026-01-01,ACO,NOX,NaN,1
4,2026-01-01,ACO,O3,NaN,1


## a) Tabla con un renglón por estación-hora y una columna por contaminante

In [6]:
tabla = df.pivot_table(index=["date", "id_station"], columns="id_parameter", values="valor")
print(tabla.shape)
tabla.head()

(133822, 9)


id_parameter             CO   NO   NO2   NOX    O3  PM10  PM2.5  PMCO   SO2
date       id_station                                                      
2026-01-01 AJM         0.74  2.0  26.0  28.0  19.0  67.0   53.0  14.0  28.0
           ATI         0.70  2.0  21.0  23.0  18.0  90.0    NaN   NaN  22.0
           BJU         1.07  NaN  32.0   NaN  17.0  62.0   41.0  21.0   3.0
           CAM          NaN  7.0  44.0  52.0   4.0  57.0   49.0   8.0  25.0
           CCA         1.12  1.0  32.0  33.0  13.0   NaN   27.0   NaN   3.0

## b) Covarianza y correlación de CO con NOX. ¿Son distintas?

In [7]:
cov_co_nox = tabla["CO"].cov(tabla["NOX"])
corr_co_nox = tabla["CO"].corr(tabla["NOX"])
print("Covarianza:", cov_co_nox)
print("Correlación:", corr_co_nox)

Covarianza: 11.586725751371082
Correlación: 0.8656303915950114


**Respuesta:** Sí, son distintas. La covarianza sale de 11.59 y la correlación de 0.87. Las dos son positivas, o sea que cuando sube el CO también sube el NOX, pero la covarianza depende de las unidades (el CO está en ppm y el NOX en ppb) y no tiene un límite, así que su tamaño no me dice qué tan fuerte es la relación. La correlación está estandarizada entre -1 y 1, y con 0.87 sí puedo decir que la relación lineal es fuerte.

## c) Eigenvalores de la matriz de correlación, con y sin faltantes

In [8]:
corr_con = tabla.corr()
corr_sin = tabla.dropna().corr()
print("Renglones completos:", len(tabla.dropna()), "de", len(tabla))

eig_con = np.linalg.eigvalsh(corr_con.values)[::-1]
eig_sin = np.linalg.eigvalsh(corr_sin.values)[::-1]
pd.DataFrame({"con faltantes": eig_con, "sin faltantes": eig_sin})

Renglones completos: 17955 de 133822


,con faltantes,sin faltantes
0,4.303344,4.455667
1,1.992863,1.978269
2,0.918607,0.912216
3,0.657686,0.644521
4,0.558406,0.552580
5,0.428957,0.334203
6,0.143673,0.122036
7,0.000074,0.000378
8,-0.003612,0.000131


**Respuesta:** Con faltantes, cada correlación se calcula con los pares que hay disponibles, y el eigenvalor más chico sale negativo (-0.0036). Eso significa que la matriz no es semidefinida positiva, así que no es una matriz de correlación válida. Sin faltantes todos los eigenvalores son positivos, pero solo quedan 17,955 de 133,822 renglones. En los dos casos los dos últimos eigenvalores están casi en 0, lo que indica combinaciones casi exactas entre variables: NOX ≈ NO + NO2 y PM10 = PM2.5 + PMCO.

## d) PM2.5 contra los demás: R² y VIF. Luego solo con gases

In [9]:
def ajustar(datos, y, xs):
    d = datos[[y] + xs].dropna()
    X = sm.add_constant(d[xs])
    modelo = sm.OLS(d[y], X).fit()
    vif = pd.Series(
        [variance_inflation_factor(X.values, i) for i in range(1, X.shape[1])],
        index=xs
    )
    return modelo, vif

In [10]:
otros = [c for c in tabla.columns if c != "PM2.5"]
m_todos, vif_todos = ajustar(tabla, "PM2.5", otros)
print("R² (todos):", m_todos.rsquared)
print(vif_todos)

R² (todos): 0.9983098229638294
CO         5.710228
NO      2324.236785
NO2      780.733295
NOX     4528.146814
O3         1.773441
PM10       4.543673
PMCO       3.403099
SO2        1.109554
dtype: float64


In [11]:
gases = ["CO", "NO", "NO2", "NOX", "O3", "SO2"]
m_gases, vif_gases = ajustar(tabla, "PM2.5", gases)
print("R² (solo gases):", m_gases.rsquared)
print(vif_gases)

R² (solo gases): 0.3166950992505001
CO        4.230546
NO     2343.290268
NO2     680.407946
NOX    4233.452205
O3        1.339026
SO2       1.039904
dtype: float64


**Respuesta:** Con todas las variables, el R² es de 0.998, pero es engañoso: PM10 es literalmente PM2.5 + PMCO, así que el modelo prácticamente "ve" la respuesta. Al dejar solo los gases, el R² baja a 0.32. En los dos modelos los VIF de NO, NO2 y NOX siguen enormes (más de 600, y hasta 4,500 en NOX) porque NOX es casi la suma de los otros dos. CO, O3 y SO2 tienen VIF aceptables.

## e) Matriz H con una estación y una semana: H² = H y traza = p
Luego el leverage de todos sin construir H.

In [12]:
semana = (tabla.xs("CCA", level="id_station")
              .loc["2026-01-01":"2026-01-07", ["PM2.5"] + gases]
              .dropna())
Xs = sm.add_constant(semana[gases]).values
H = Xs @ np.linalg.pinv(Xs.T @ Xs) @ Xs.T

print("n, p:", Xs.shape)
print("H² = H:", np.allclose(H @ H, H))
print("traza(H):", np.trace(H), " p:", Xs.shape[1])

n, p: (165, 7)
H² = H: True
traza(H): 7.000000000002947  p: 7


In [13]:
X = m_gases.model.exog
Q, R = np.linalg.qr(X)
leverage = (Q ** 2).sum(axis=1)

print("Suma de leverages:", leverage.sum(), " p:", X.shape[1])
print("Igual a statsmodels:", np.allclose(leverage, m_gases.get_influence().hat_matrix_diag))

Suma de leverages: 7.0  p: 7
Igual a statsmodels: True


## f) Distancia de Cook: ¿los 10 más influyentes son los de más leverage? Reajustar sin ellos

In [14]:
idx = m_gases.model.data.row_labels
cook = pd.Series(m_gases.get_influence().cooks_distance[0], index=idx)
lev = pd.Series(leverage, index=idx)

top_cook = cook.nlargest(10).index
top_lev = lev.nlargest(10).index

pd.DataFrame({
    "cook": cook[top_cook],
    "leverage": lev[top_cook],
    "rank leverage": lev.rank(ascending=False)[top_cook].astype(int)
})

,,cook,leverage,rank leverage
date,id_station,,,
2026-01-01 07:00:00,SAC,0.261088,0.002685,122
2026-01-01 03:00:00,SAC,0.253419,0.002230,189
2026-01-01 08:00:00,SAC,0.194491,0.002592,137
2026-01-01 02:00:00,SAC,0.178968,0.001858,257
2026-01-01 04:00:00,SAC,0.122915,0.001584,315
2026-01-01 05:00:00,NEZ,0.096208,0.001715,280
2026-01-01 06:00:00,SAC,0.080198,0.003611,70
2026-01-01 09:00:00,SAC,0.074228,0.000906,796
2026-01-01 06:00:00,NEZ,0.069010,0.000846,874


In [15]:
print("En común entre top 10 Cook y top 10 leverage:", len(set(top_cook) & set(top_lev)))

En común entre top 10 Cook y top 10 leverage: 0


In [16]:
d = tabla[["PM2.5"] + gases].dropna().drop(top_cook)
m_sin = sm.OLS(d["PM2.5"], sm.add_constant(d[gases])).fit()

pd.DataFrame({
    "con todos": m_gases.params,
    "sin top 10 Cook": m_sin.params
}).assign(R2_con=m_gases.rsquared, R2_sin=m_sin.rsquared)

,con todos,sin top 10 Cook,R2_con,R2_sin
const,1.586491,1.614462,0.316695,0.342393
CO,11.531709,8.690767,0.316695,0.342393
NO,-0.074357,-0.028392,0.316695,0.342393
NO2,0.200403,0.281031,0.316695,0.342393
NOX,0.052246,0.020886,0.316695,0.342393
O3,0.165720,0.169183,0.316695,0.342393
SO2,0.355599,0.350826,0.316695,0.342393


**Respuesta:** No. Ninguno de los 10 puntos con mayor distancia de Cook está en el top 10 de leverage; en leverage quedan entre el lugar 70 y el 874. Los 10 son de la madrugada del 1 de enero (de la 1 a las 9 h), casi todos en SAC y un par en NEZ, cuando el PM2.5 llegó a 200–349 µg/m³ por los fuegos artificiales de Año Nuevo, mientras que la mediana es de 19. Los gases no explican esos picos, así que la influencia viene de residuos enormes y no de valores raros en X. Cook combina las dos cosas, por eso un punto puede ser muy influyente sin tener mucho leverage. Al reajustar sin esos 10 puntos (de 32,686), el R² sube de 0.317 a 0.342, el coeficiente del CO baja de 11.5 a 8.7 y el del NO2 sube de 0.20 a 0.28. Con solo 10 horas el coeficiente del CO cambia como un 25%.